# GPU Image Editing Backend — FLUX.1-Fill-dev (with SDXL Inpainting fallback)

This notebook auto-selects the **best available diffusion model** for general-purpose inpainting:

| Model | Quality | Requires |
|---|---|---|
| **FLUX.1-Fill-dev** (preferred) | State of the art | HF account + accept license + HF_TOKEN secret |
| **SDXL Inpainting 1.0** (fallback) | Very good — general purpose | Nothing — fully open, MIT license |

The notebook tries FLUX first. If your `HF_TOKEN` is missing or the license isn't accepted, it **automatically loads SDXL Inpainting** — no intervention needed.

## API contract (unchanged)
Local app calls `GET /health`, `POST /v1/mask`, `POST /v1/edit` with `X-Backend-Token` auth. Both models serve identical routes.

## Before running
1. Enable a **T4 GPU** + internet access in Kaggle settings.
2. Add Kaggle Secrets: `NGROK_AUTHTOKEN`, `GPU_BACKEND_TOKEN`.
3. **Optional (for FLUX)**: Accept the license at [huggingface.co/black-forest-labs/FLUX.1-Fill-dev](https://huggingface.co/black-forest-labs/FLUX.1-Fill-dev) → click **"Agree and access repository"**. Then add `HF_TOKEN` as a Kaggle Secret (get your token at [huggingface.co/settings/tokens](https://huggingface.co/settings/tokens)).
4. Run all cells top to bottom.

## 1. Install dependencies

In [ ]:
%pip uninstall -y torchao 2>/dev/null; true
%pip -q install \
  "diffusers>=0.30.0,<0.42" \
  "transformers>=4.45,<5.1" \
  "accelerate>=1.0" \
  "sentencepiece>=0.2" \
  safetensors \
  fastapi \
  "uvicorn[standard]" \
  python-multipart \
  pyngrok \
  pillow \
  opencv-python-headless \
  huggingface_hub \
  requests \
  scipy \
  "protobuf>=3.20"
print("Dependencies installed.")

## 2. GPU pre-flight and secrets

In [ ]:
import torch, os
from getpass import getpass

print("CUDA available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("GPU unavailable. Enable a GPU accelerator in Kaggle settings.")

for i in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(i)
    free, total = torch.cuda.mem_get_info(i)
    print(f"GPU {i}: {props.name} | total VRAM {total/1024**3:.1f} GiB | free VRAM {free/1024**3:.1f} GiB")

PORT = 8001

def get_secret(name, required=True):
    value = None
    try:
        from kaggle_secrets import UserSecretsClient
        value = UserSecretsClient().get_secret(name)
    except Exception:
        value = None
    if not value and required:
        value = getpass(f"Enter {name} (input hidden): ").strip()
    if required and not value:
        raise RuntimeError(f"Missing required secret: {name}")
    return value or ""

NGROK_AUTHTOKEN = get_secret("NGROK_AUTHTOKEN")
GPU_BACKEND_TOKEN = get_secret("GPU_BACKEND_TOKEN")
HF_TOKEN = get_secret("HF_TOKEN", required=False)
os.environ["HF_TOKEN"] = HF_TOKEN or ""

if HF_TOKEN:
    print("HF_TOKEN found — will attempt FLUX.1-Fill-dev (best quality).")
else:
    print("HF_TOKEN not set — will use SDXL Inpainting 1.0 (no token needed).")
    print("To unlock FLUX: accept license at https://huggingface.co/black-forest-labs/FLUX.1-Fill-dev")
    print("then add HF_TOKEN to Kaggle Secrets and rerun.")
print("Required secrets loaded.")

## 3. Load the best available diffusion model

This cell tries **FLUX.1-Fill-dev** first (state of the art, needs HF token + license accepted).  
If that fails for any reason (gated, no token, OOM), it **automatically falls back to SDXL Inpainting 1.0** — fully open, MIT licensed, no token needed, always works.

> No manual intervention needed — just run the cell.

In [ ]:
import torch, gc
from diffusers import FluxFillPipeline, StableDiffusionXLInpaintPipeline

# ── Model options ─────────────────────────────────────────────────────────────
# Option A: FLUX.1-Fill-dev — state of the art, gated (needs HF token + license)
FLUX_MODEL_ID  = "black-forest-labs/FLUX.1-Fill-dev"
# Option B: Official SDXL inpainting pipeline — fully open, MIT license, no token
# This is the Stability AI reference inpainting pipeline in proper diffusers format.
SDXL_MODEL_ID  = "diffusers/stable-diffusion-xl-1.0-inpainting-0.1"

pipe       = None
MODEL_NAME = None
USE_FLUX   = False


def _load_flux():
    global pipe, MODEL_NAME, USE_FLUX
    print(f"[FLUX] Attempting to load {FLUX_MODEL_ID} ...")
    print("  Downloading ~24 GB — this takes 3-6 min on first run.")
    _pipe = FluxFillPipeline.from_pretrained(
        FLUX_MODEL_ID,
        dtype=torch.bfloat16,          # use 'dtype' (new API)
        token=HF_TOKEN or None,
    )
    _pipe.enable_model_cpu_offload()
    try:  _pipe.enable_attention_slicing()
    except Exception: pass
    try:  _pipe.vae.enable_tiling()
    except Exception: pass
    pipe       = _pipe
    MODEL_NAME = "FLUX.1-Fill-dev"
    USE_FLUX   = True
    print(f"[FLUX] Loaded successfully — {type(pipe).__name__}")


def _load_sdxl():
    """Load the official SDXL Inpainting 1.0 pipeline directly.

    This pipeline IS in proper diffusers multi-folder format and can be
    loaded with a single from_pretrained call — no UNet swapping needed.
    It is MIT licensed and requires no HuggingFace token.
    """
    global pipe, MODEL_NAME, USE_FLUX
    print(f"[SDXL] Loading {SDXL_MODEL_ID} ...")
    print("  Downloading ~7 GB — this takes 1-2 min on first run.")
    _pipe = StableDiffusionXLInpaintPipeline.from_pretrained(
        SDXL_MODEL_ID,
        torch_dtype=torch.float16,
        use_safetensors=True,
        variant="fp16",
    )
    _pipe.enable_model_cpu_offload()
    try:  _pipe.enable_attention_slicing()
    except Exception: pass
    pipe       = _pipe
    MODEL_NAME = "SDXL-Inpainting-1.0"
    USE_FLUX   = False
    print(f"[SDXL] Loaded successfully — {type(pipe).__name__}")


# ── Auto-selection logic ──────────────────────────────────────────────────────

def _try_flux_then_sdxl():
    try:
        _load_flux()
    except Exception as flux_err:
        err_str = str(flux_err)
        print()
        if "403" in err_str or "GatedRepo" in err_str or "gated" in err_str.lower():
            print("=" * 62)
            print("  FLUX ACCESS DENIED (403 / GatedRepoError)")
            print("=" * 62)
            print("  Your HF_TOKEN exists but you haven't accepted the license yet.")
            print("  Fix (optional, for best quality):")
            print("    1. Go to https://huggingface.co/black-forest-labs/FLUX.1-Fill-dev")
            print("    2. Click 'Agree and access repository' while logged in.")
            print("    3. Rerun this notebook.")
            print()
        else:
            print(f"  FLUX load failed ({type(flux_err).__name__}): {str(flux_err)[:200]}")
        print("  → Falling back to SDXL Inpainting 1.0 (fully open, MIT license)")
        print("=" * 62)
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        _load_sdxl()


if HF_TOKEN:
    _try_flux_then_sdxl()
else:
    _load_sdxl()


gc.collect()
torch.cuda.empty_cache()
free, total = torch.cuda.mem_get_info(0)
print()
print(f"Active model : {MODEL_NAME}")
print(f"VRAM         : {free/1024**3:.1f} GiB free / {total/1024**3:.1f} GiB total")

## 4. Load Grounding DINO + SAM for automatic masking

Kept on CPU to preserve GPU memory for diffusion.

In [ ]:
from transformers import AutoProcessor, AutoModelForZeroShotObjectDetection, SamModel, SamProcessor

GROUNDING_DINO_ID = "IDEA-Research/grounding-dino-tiny"
SAM_ID = "facebook/sam-vit-base"

print("Loading Grounding DINO on CPU...")
grounding_processor = AutoProcessor.from_pretrained(GROUNDING_DINO_ID, token=HF_TOKEN or None)
grounding_model = AutoModelForZeroShotObjectDetection.from_pretrained(
    GROUNDING_DINO_ID, token=HF_TOKEN or None
).to("cpu").eval()

print("Loading SAM on CPU...")
sam_processor = SamProcessor.from_pretrained(SAM_ID, token=HF_TOKEN or None)
sam_model = SamModel.from_pretrained(SAM_ID, token=HF_TOKEN or None).to("cpu").eval()
print("Automatic masking models loaded (CPU).")

## 5. Smoke test — verify the pipeline runs before exposing the API

In [ ]:
from PIL import Image, ImageDraw
import numpy as np

# Simple 512x512 test: inpaint a grey square on a neutral background.
smoke_src  = Image.new("RGB", (512, 512), (190, 185, 180))
draw = ImageDraw.Draw(smoke_src)
draw.rectangle((160, 140, 350, 380), fill=(80, 80, 80))
smoke_mask = Image.new("L",  (512, 512), 0)
mask_draw  = ImageDraw.Draw(smoke_mask)
mask_draw.rectangle((150, 130, 360, 390), fill=255)

print(f"Running smoke test with {MODEL_NAME} ...")

with torch.inference_mode():
    if USE_FLUX:
        smoke_out = pipe(
            prompt="a smooth wooden cube, studio lighting, photorealistic",
            image=smoke_src,
            mask_image=smoke_mask,
            height=512, width=512,
            num_inference_steps=20,
            guidance_scale=30.0,
            generator=torch.Generator(device="cpu").manual_seed(42),
        ).images[0]
    else:
        smoke_out = pipe(
            prompt="a smooth wooden cube, studio photograph, photorealistic",
            negative_prompt="blurry, cartoon, low quality, distorted",
            image=smoke_src,
            mask_image=smoke_mask,
            num_inference_steps=20,
            guidance_scale=7.5,
            strength=0.95,
            generator=torch.Generator(device="cpu").manual_seed(42),
        ).images[0]

assert smoke_out.size in ((512, 512), (1024, 1024)), f"Unexpected output size: {smoke_out.size}"
smoke_out = smoke_out.resize(smoke_src.size, Image.Resampling.LANCZOS)
print(f"Smoke test PASSED — {MODEL_NAME} is working correctly.")
display(smoke_out)

## 6. Define FastAPI endpoints

In [ ]:
import io, re, secrets, threading, gc
import numpy as np
import cv2
from PIL import Image
from fastapi import FastAPI, File, Form, Header, HTTPException, UploadFile
from fastapi.responses import Response

app = FastAPI(title="GPU Image Editing Backend", version="3.2.0-sdxl-fallback")
_inference_lock = threading.Lock()

GENERIC_TARGETS = {"", "object", "thing", "item", "the object", "object described in prompt"}


# ── Auth ──────────────────────────────────────────────────────────────────────

def _check_token(x_backend_token):
    if not x_backend_token or not secrets.compare_digest(x_backend_token, GPU_BACKEND_TOKEN):
        raise HTTPException(status_code=401, detail="Missing or invalid backend token")

def _open_rgb(raw, name):
    try:
        return Image.open(io.BytesIO(raw)).convert("RGB")
    except Exception as exc:
        raise HTTPException(status_code=400, detail=f"Invalid {name} image") from exc

def _open_mask(raw):
    try:
        return Image.open(io.BytesIO(raw)).convert("L")
    except Exception as exc:
        raise HTTPException(status_code=400, detail="Invalid mask image") from exc


# ── Target normalisation ──────────────────────────────────────────────────────

def _normalise_target(target, prompt, operation):
    raw = (target or "").strip().strip(" .")
    if raw.lower() not in GENERIC_TARGETS:
        return raw[:160]
    text = (prompt or "").strip()
    patterns = [
        r"\badd\s+.+?\s+to\s+(?:the\s+)?(.+)$",
        r"\badd\s+.+?\s+(?:over|onto|on|around|across|above|under|beneath)\s+(?:the\s+)?(.+)$",
        r"\bput\s+.+?\s+on\s+(?:the\s+)?(.+)$",
        r"\bplace\s+.+?\s+on\s+(?:the\s+)?(.+)$",
        r"\b(?:remove|delete|erase)\s+(?:the|a|an)?\s*(.+)$",
        r"\b(?:replace|change|turn|paint|color|colour)\s+(?:the|a|an)?\s*(.+?)\s+(?:with|into|to)\s+.+$",
        r"\bmake\s+(?:the|a|an)?\s*(.+?)\s+(?:red|blue|green|yellow|black|white|brown|pink|purple|orange|silver|gold|wooden|metallic)\b",
    ]
    for pattern in patterns:
        m = re.search(pattern, text, flags=re.IGNORECASE)
        if m:
            c = m.group(1).strip(" .,!?:;")
            if c:
                return c[:160]
    return text[:160]


# ── Grounding DINO detection ──────────────────────────────────────────────────

def _dino_detect(src, phrase):
    text = phrase.lower().strip()
    if not text.endswith("."):
        text += "."
    det_inputs = grounding_processor(images=src, text=text, return_tensors="pt")
    model_device = next(grounding_model.parameters()).device
    det_inputs = {k: v.to(model_device) if torch.is_tensor(v) else v for k, v in det_inputs.items()}
    with torch.inference_mode():
        det_outputs = grounding_model(**det_inputs)
    fn = grounding_processor.post_process_grounded_object_detection
    try:
        return fn(det_outputs, input_ids=det_inputs.get("input_ids"),
                  box_threshold=0.22, text_threshold=0.15,
                  target_sizes=[(src.height, src.width)])[0]
    except TypeError:
        return fn(det_outputs, threshold=0.22, text_threshold=0.15,
                  target_sizes=[(src.height, src.width)])[0]


# ── Automatic masking: Grounding DINO + SAM ───────────────────────────────────

def _automatic_mask(src, target, prompt, operation="modify"):
    phrase = _normalise_target(target, prompt, operation)
    if not phrase:
        raise HTTPException(status_code=422, detail="Provide a target description or upload a mask.")

    low = phrase.lower()
    candidates = [phrase]
    if any(t in low for t in ("face","eye","glasses","sunglasses","hat","earring","hair","beard","mustache")):
        candidates += ["face", "head", "person"]
    if "head" in low:  candidates += ["head", "face", "person"]
    if "wall" in low:  candidates += ["wall", "building wall"]
    words = phrase.split()
    if len(words) > 5:
        candidates.append(" ".join(words[:5]))
    candidates = list(dict.fromkeys(c.strip(" .") for c in candidates if c.strip(" .")))

    failures = []
    for candidate in candidates:
        try:
            result = _dino_detect(src, candidate)
            boxes  = result.get("boxes")
            scores = result.get("scores")
            if boxes is None or len(boxes) == 0:
                failures.append(f"{candidate}: no detection")
                continue

            order = torch.argsort(scores, descending=True).tolist() if (scores is not None and len(scores)) else list(range(len(boxes)))

            for box_idx in order[:4]:
                box = boxes[box_idx].detach().cpu().tolist()
                x1, y1, x2, y2 = [max(0.0, float(v)) for v in box]
                x2 = min(float(src.width), x2)
                y2 = min(float(src.height), y2)
                if x2 - x1 < 4 or y2 - y1 < 4:
                    continue

                sam_inputs = sam_processor(images=src, input_boxes=[[[x1, y1, x2, y2]]], return_tensors="pt")
                sam_dev = next(sam_model.parameters()).device
                sam_inputs = {k: v.to(sam_dev) if torch.is_tensor(v) else v for k, v in sam_inputs.items()}
                with torch.inference_mode():
                    sam_outputs = sam_model(**sam_inputs)

                processed = sam_processor.post_process_masks(
                    sam_outputs.pred_masks.detach().cpu(),
                    sam_inputs["original_sizes"].cpu(),
                    sam_inputs["reshaped_input_sizes"].cpu(),
                )[0]

                cand_masks = processed[0] if processed.ndim == 4 else processed
                iou = sam_outputs.iou_scores.detach().cpu()
                cscores = iou[0, 0] if iou.ndim == 3 else (iou[0] if iou.ndim == 2 else iou)

                best = int(torch.argmax(cscores).item())
                sel  = cand_masks[best]
                marr = sel.numpy() if sel.dtype == torch.bool else (sel.numpy() > 0)

                is_glasses = any(k in phrase.lower() for k in ("sunglass", "glasses", "shades", "spectacles", "eyewear"))
                if is_glasses and candidate in ("face", "head", "person"):
                    fw = x2 - x1
                    fh = y2 - y1
                    ex1 = max(0.0, x1 - fw * 0.04)
                    ex2 = min(float(src.width), x2 + fw * 0.04)
                    ey1 = max(0.0, y1 + fh * 0.20)
                    ey2 = min(float(src.height), y1 + fh * 0.52)
                    mimg = Image.new("L", src.size, 0)
                    mdraw = ImageDraw.Draw(mimg)
                    mdraw.rounded_rectangle((int(ex1), int(ey1), int(ex2), int(ey2)), radius=int(fh * 0.08), fill=255)
                    print(f"Auto-mask (sunglasses eye-band): region=({int(ex1)}, {int(ey1)}, {int(ex2)}, {int(ey2)})")
                    return mimg, candidate

                if marr.ndim != 2 or not marr.any():
                    failures.append(f"{candidate}: empty mask")
                    continue

                mu8  = marr.astype(np.uint8) * 255
                mu8  = cv2.dilate(mu8, np.ones((5, 5), np.uint8), iterations=1)
                mimg = Image.fromarray(mu8, mode="L")
                if mimg.size != src.size:
                    mimg = mimg.resize(src.size, Image.Resampling.NEAREST)

                print(f"Auto-mask: target={candidate!r}, box={box_idx}, mask={best}, IoU={float(cscores[best]):.3f}")
                return mimg, candidate

            failures.append(f"{candidate}: no usable SAM mask")
        except HTTPException:
            raise
        except Exception as exc:
            failures.append(f"{candidate}: {type(exc).__name__}: {str(exc)[:100]}")

    raise HTTPException(
        status_code=422,
        detail=f"Could not mask '{phrase}'. Details: {'; '.join(failures[-6:])}. Try uploading a mask."
    )


# ── Inference size helper ─────────────────────────────────────────────────────

def _inference_size(width, height, max_side=1024):
    scale = min(max_side / max(width, height), 1.0)
    def r(v): return max(64, int(round((v * scale) / 8.0)) * 8)
    return r(width), r(height)


# ── PNG response ──────────────────────────────────────────────────────────────

def _png_response(image, headers=None):
    buf = io.BytesIO()
    image.save(buf, format="PNG")
    return Response(content=buf.getvalue(), media_type="image/png", headers=headers or {})


print(f"FastAPI helpers defined. Active model: {MODEL_NAME}")

In [ ]:
# ── Endpoints ─────────────────────────────────────────────────────────────────

@app.get("/health")
def health(x_backend_token: str = Header(default=None)):
    _check_token(x_backend_token)
    return {
        "status": "ok",
        "service": "gpu-image-editing-backend",
        "model": MODEL_NAME,
        "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none",
        "model_loaded": pipe is not None,
        "automatic_masking": grounding_model is not None and sam_model is not None,
        "endpoints": ["/health", "/v1/mask", "/v1/edit"],
    }


@app.post("/v1/mask")
async def create_mask(
    image: UploadFile = File(...),
    target: str = Form("object"),
    prompt: str = Form(""),
    operation: str = Form("modify"),
    x_backend_token: str = Header(default=None),
):
    _check_token(x_backend_token)
    src = _open_rgb(await image.read(), "source")
    if src.width < 64 or src.height < 64:
        raise HTTPException(status_code=400, detail="Image must be at least 64x64")
    mask_img, selected_target = _automatic_mask(src, target, prompt, operation)
    return _png_response(mask_img, {"X-Mask-Mode": "automatic", "X-Mask-Target": selected_target})


@app.post("/v1/edit")
async def edit_image(
    image: UploadFile = File(...),
    mask: UploadFile = File(None),
    prompt: str = Form(...),
    target: str = Form("object"),
    operation: str = Form("modify"),
    auto_mask: bool = Form(True),
    x_backend_token: str = Header(default=None),
    num_inference_steps: int = Form(28),
    guidance_scale: float = Form(30.0),
    controlnet_conditioning_scale: float = Form(0.0),  # accepted but unused
    strength: float = Form(0.95),
    seed: int = Form(42),
):
    """
    General-purpose image editing:
      image + text prompt -> auto/manual mask -> diffusion inpainting -> PNG

    Works with both FLUX.1-Fill-dev and SDXL Inpainting 1.0.
    FLUX: guidance_scale 28-35, strength ignored.
    SDXL: guidance_scale auto-remapped from FLUX range to 7-9, strength 0.85-0.98.
    """
    _check_token(x_backend_token)

    if not prompt or not prompt.strip():
        raise HTTPException(status_code=400, detail="prompt must not be empty")
    if not 1 <= num_inference_steps <= 60:
        raise HTTPException(status_code=400, detail="num_inference_steps must be 1-60")
    if operation not in {"modify", "replace", "add", "remove"}:
        raise HTTPException(status_code=400, detail="operation must be modify/replace/add/remove")

    src = _open_rgb(await image.read(), "source")
    if src.width < 64 or src.height < 64:
        raise HTTPException(status_code=400, detail="Image must be at least 64x64")

    # Obtain mask
    if mask is not None:
        mask_bytes = await mask.read()
        if not mask_bytes:
            raise HTTPException(status_code=400, detail="Uploaded mask is empty")
        mask_img = _open_mask(mask_bytes)
        if mask_img.size != src.size:
            mask_img = mask_img.resize(src.size, Image.Resampling.NEAREST)
        selected_target = target or "manual mask"
        mask_mode = "provided"
    elif auto_mask:
        mask_img, selected_target = _automatic_mask(src, target, prompt, operation)
        mask_mode = "automatic"
    else:
        raise HTTPException(status_code=422, detail="No mask supplied. Upload a mask or enable auto_mask.")

    mask_arr_native = np.asarray(mask_img) > 127
    if not mask_arr_native.any():
        raise HTTPException(status_code=422, detail="Mask has no editable (white) pixels")
    if mask_arr_native.all():
        raise HTTPException(status_code=422, detail="Mask covers entire image; pick a smaller region.")

    # Build photorealistic prompt
    if operation == "remove":
        prompt_full = (
            f"Seamlessly remove the {selected_target} and reconstruct the background naturally. "
            "Photorealistic, consistent lighting, texture, and perspective."
        )
        neg_prompt = "object remaining, artifact, blur, distortion"
    else:
        p_clean = prompt.strip()
        if "photorealistic" not in p_clean.lower(): p_clean += ", photorealistic, sharp focus"
        prompt_full = p_clean
        neg_prompt  = "cartoon, illustration, CGI, blurry, low quality, watermark, text, distorted"

    # Scale to model-appropriate resolution
    max_side = 1024 if USE_FLUX else 768
    work_w, work_h = _inference_size(src.width, src.height, max_side=max_side)
    src_work  = src.resize((work_w, work_h), Image.Resampling.LANCZOS)
    mask_work = mask_img.resize((work_w, work_h), Image.Resampling.NEAREST)

    try:
        with _inference_lock, torch.inference_mode():
            if USE_FLUX:
                result = pipe(
                    prompt=prompt_full,
                    image=src_work,
                    mask_image=mask_work,
                    height=work_h, width=work_w,
                    num_inference_steps=num_inference_steps,
                    guidance_scale=min(max(guidance_scale, 1.0), 50.0),
                    generator=torch.Generator(device="cpu").manual_seed(int(seed)),
                ).images[0].convert("RGB")
            else:
                # SDXL inpainting: remap FLUX-style guidance (28-35) to SDXL range (7-9)
                sdxl_guidance = 7.5 if guidance_scale > 12.0 else min(max(guidance_scale, 1.0), 12.0)
                result = pipe(
                    prompt=prompt_full,
                    negative_prompt=neg_prompt,
                    image=src_work,
                    mask_image=mask_work,
                    num_inference_steps=num_inference_steps,
                    guidance_scale=sdxl_guidance,
                    strength=min(max(strength, 0.0), 1.0),
                    generator=torch.Generator(device="cpu").manual_seed(int(seed)),
                ).images[0].convert("RGB")

        # Restore original pixels outside the mask (exact pixel preservation)
        generated_native = result.resize(src.size, Image.Resampling.LANCZOS)
        original  = np.asarray(src)
        generated = np.asarray(generated_native)
        output    = original.copy()
        output[mask_arr_native] = generated[mask_arr_native]
        response_image = Image.fromarray(output)

        gc.collect()
        torch.cuda.empty_cache()

        return _png_response(
            response_image,
            {
                "X-Seed":       str(seed),
                "X-Mask-Mode":  mask_mode,
                "X-Mask-Target": selected_target[:120],
                "X-Operation":  operation,
                "X-Model":      MODEL_NAME,
            },
        )

    except HTTPException:
        raise
    except torch.cuda.OutOfMemoryError as exc:
        gc.collect(); torch.cuda.empty_cache()
        raise HTTPException(status_code=503,
            detail="GPU OOM. Retry with a smaller image or fewer steps.") from exc
    except Exception as exc:
        raise HTTPException(status_code=500,
            detail=f"Inference failed ({type(exc).__name__}): {str(exc)[:300]}") from exc


print("=" * 70)
print(f"GPU Backend — routes defined | Model: {MODEL_NAME}")
print("Routes: GET /health | POST /v1/mask | POST /v1/edit")
print("=" * 70)

## 7. Start FastAPI + ngrok tunnel

In [ ]:
import os, time, threading, urllib.request, urllib.error
import uvicorn
from pyngrok import ngrok, conf
from kaggle_secrets import UserSecretsClient

NGROK_AUTHTOKEN = UserSecretsClient().get_secret("NGROK_AUTHTOKEN").strip()
if not NGROK_AUTHTOKEN or any(ch.isspace() for ch in NGROK_AUTHTOKEN):
    raise RuntimeError("NGROK_AUTHTOKEN is empty or contains whitespace.")

try:
    os.environ["GPU_BACKEND_TOKEN"] = UserSecretsClient().get_secret("GPU_BACKEND_TOKEN").strip()
except Exception:
    pass

try: ngrok.kill()
except Exception: pass
ngrok.set_auth_token(NGROK_AUTHTOKEN)
conf.get_default().auth_token = NGROK_AUTHTOKEN

if "uvicorn_thread" in globals() and uvicorn_thread.is_alive():
    print("Uvicorn already running; reusing.")
else:
    uvicorn_config = uvicorn.Config(app, host="0.0.0.0", port=PORT, log_level="warning", access_log=False)
    uvicorn_server = uvicorn.Server(uvicorn_config)
    uvicorn_thread = threading.Thread(target=uvicorn_server.run, daemon=True, name="gpu-backend-uvicorn")
    uvicorn_thread.start()

ready = False
for _ in range(60):
    try:
        urllib.request.urlopen(urllib.request.Request(f"http://127.0.0.1:{PORT}/health"), timeout=1)
        ready = True; break
    except urllib.error.HTTPError as exc:
        if exc.code == 401: ready = True; break
        time.sleep(1)
    except Exception:
        time.sleep(1)

if not ready:
    raise RuntimeError("Uvicorn did not become ready. Check earlier cell output.")

tunnel = ngrok.connect(PORT)
PUBLIC_URL = tunnel.public_url
if not PUBLIC_URL.startswith("https://"):
    raise RuntimeError("Expected an HTTPS URL from ngrok.")

print("=" * 70)
print(f"GPU backend started  |  Model: {MODEL_NAME}")
print(f"Public HTTPS base URL : {PUBLIC_URL}")
print(f"Health endpoint       : {PUBLIC_URL}/health")
print(f"Edit endpoint         : {PUBLIC_URL}/v1/edit")
print()
print("→ Paste into your local .env:")
print(f"  GPU_BACKEND_URL={PUBLIC_URL}")
print("Keep this Kaggle session running. Never share the backend token.")
print("=" * 70)

## 8. Verify authenticated health endpoint

In [ ]:
import requests
resp = requests.get(PUBLIC_URL + "/health", headers={"X-Backend-Token": GPU_BACKEND_TOKEN}, timeout=30)
print("HTTP status:", resp.status_code)
resp.raise_for_status()
print(resp.json())

## 9. What you can do now

| Request | How the model handles it |
|---|---|
| Add sunglasses | Auto-masks the face region, inpaints realistic sunglasses |
| Remove an object | Auto-masks the object, reconstructs background seamlessly |
| Change clothing colour | Auto-masks the garment, recolours photorealistically |
| Add a hat | Auto-masks the head region, generates the hat |
| Change background | Inverted mask, replaces background |
| Modify facial hair | Auto-masks the face area, adjusts beard/moustache |

**To upgrade to FLUX later (optional — best quality):**
1. Visit https://huggingface.co/black-forest-labs/FLUX.1-Fill-dev
2. Click **"Agree and access repository"** (instant one-click accept)
3. Get your token at https://huggingface.co/settings/tokens
4. Add `HF_TOKEN` as a Kaggle Secret and rerun the notebook